# Notebook 2 — The SLC26 Superfamily: Subfamilies and Prestin

## Objectives
- Build a phylogenetic tree with **FastTree** and explore it with **ETE4 smartview**
- Extract **all subfamilies** from duplication events in the tree
- Build a **phylogenetic profile** (gene count per species per subfamily) on the NCBI taxonomy
- Compare **alignment statistics** and **branch lengths** across subfamilies
- Compare **phylogenetic methods** on prestin (SLC26A5)

## Recap
From Notebook 1 we have a trimmed MAFFT alignment of 297 SLC26 sequences from 30 species.

In [ ]:
import os, subprocess, io, re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

sns.set_theme(style="whitegrid")

DATA = os.path.join("..", "data")
FIGS = os.path.join("..", "figures")
FASTA = os.path.join(DATA, "selection2_clustalo.fa")
SUB_DIR = os.path.join(DATA, "subfamilies")
os.makedirs(SUB_DIR, exist_ok=True)

def read_fasta(path):
    """Read FASTA → dict {header: sequence}."""
    seqs = {}; header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                header = line[1:].split()[0]; seqs[header] = []
            elif header:
                seqs[header].append(line)
    return {h: "".join(s) for h, s in seqs.items()}

def write_fasta(seqs, path):
    """Write dict {header: sequence} → FASTA file."""
    with open(path, "w") as f:
        for h, s in seqs.items():
            f.write(f">{h}\n{s}\n")

# Load gene name mapping (for labeling extracted clades)
seqid2gene = {}
for line in open(os.path.join(DATA, "selection2.clustalo.seqid2gname.tab")):
    parts = line.strip().split("\t")
    if len(parts) >= 2:
        seqid2gene[parts[0]] = parts[1].upper()

# Trimmed alignment from Notebook 1
TRIM = os.path.join(DATA, "slc26.mafft.gt01.fa")
if not os.path.exists(TRIM):
    ALN = os.path.join(DATA, "slc26.mafft.fa")
    if not os.path.exists(ALN):
        print("Running MAFFT...")
        with open(ALN, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", FASTA],
                           stdout=f, stderr=subprocess.PIPE, check=True)
    subprocess.run(["trimal", "-in", ALN, "-out", TRIM, "-gt", "0.1", "-fasta"],
                   check=True, capture_output=True)

aln_trim = read_fasta(TRIM)
print(f"Trimmed alignment: {len(aln_trim)} sequences × {len(next(iter(aln_trim.values())))} columns")

---
## 1. Tree inference with FastTree

In [ ]:
TREE = os.path.join(DATA, "slc26.mafft.gt01.fasttree.nwk")

if not os.path.exists(TREE):
    print("Running FastTree (LG model)...")
    with open(TREE, "w") as f:
        subprocess.run(["FastTree", "-lg", TRIM], stdout=f,
                       stderr=subprocess.PIPE, check=True)
    print("Done.")
else:
    print(f"Cached: {TREE}")

---
## 2. Loading the tree in ETE4

[ETE4](https://etetoolkit.github.io/ete/) extends ETE3 with an interactive
browser-based viewer (**smartview**). Same `PhyloTree` API for annotation,
but much richer exploration.

In [ ]:
from ete4 import PhyloTree
from ete4.smartview import Layout, TextFace, SeqFace

t = PhyloTree(open(TREE).read(),
              sp_naming_function=lambda node: node.name.split('.')[0])

t.set_outgroup(t.get_midpoint_outgroup())
t.resolve_polytomy(descendants=True)

print("Annotating with NCBI taxonomy...")
t.annotate_ncbi_taxa()

events = t.get_descendant_evol_events()
n_dup = sum(1 for e in events if e.etype == "D")
n_spec = sum(1 for e in events if e.etype == "S")
print(f"Duplication events: {n_dup}")
print(f"Speciation events:  {n_spec}")
print(f"Leaves: {len(list(t.leaves()))}")

### 2.1 What species do we have?

In [ ]:
species_names = {}
for leaf in t.leaves():
    taxid = leaf.name.split(".")[0]
    sci = leaf.props.get("sci_name", taxid)
    species_names[taxid] = sci

taxid_counts = Counter(leaf.name.split(".")[0] for leaf in t.leaves())

print(f"{'TaxID':<10s} {'Species':<40s} {'Genes':>5s}")
print("-" * 60)
for taxid, count in sorted(taxid_counts.items(), key=lambda x: -x[1]):
    print(f"  {taxid:<10s} {species_names.get(taxid, '?'):<40s} {count:>5d}")

### ✏️ Exercise 1

Which species are **bats** (Chiroptera)? Which echolocate?
Which are **cetaceans**? Toothed (echolocating) or baleen?

**Hint:** All Chiroptera except Pteropodidae echolocate. All Odontoceti echolocate.
```python
from ete3 import NCBITaxa
ncbi = NCBITaxa()
lineage = ncbi.get_lineage(9739)
print([ncbi.get_taxid_translator(lineage)[t] for t in lineage])
```

---
## 3. Interactive exploration (ETE4 smartview)

In [ ]:
def node_names_style(node):
    if node.is_leaf:
        acc = node.name.split(".")[1] if "." in node.name else node.name
        sci = node.props.get("sci_name", "")
        return [
            TextFace(acc, style={"fill": "black"}, column=0, position="right"),
            TextFace(f"({sci})", style={"fill": "gray"}, column=1, position="right"),
        ]

def node_evo_style(node):
    if node.props.get("evoltype") == "D":
        return {"dot": {"shape": "triangle", "radius": 8, "fill": "red"}}

layouts = [
    Layout(name="Tree", draw_tree={"collapse_size": 1, "smartzoom": False}, active=True),
    Layout(name="Names", draw_node=node_names_style),
    Layout(name="Duplications", draw_node=node_evo_style, active=True),
]

print("Launching ETE4 smartview...")
print("  Red triangles = duplication events (subfamily boundaries)")
t.explore(layouts=layouts, keep_server=True, port=5006,
          show_leaf_name=False,
          include_props=("name", "sci_name", "evoltype", "dist", "support"))

---
## 4. Extracting subfamilies from tree topology

The SLC26 subfamilies (A1–A11) arose by ancient gene duplications.
We extract them by walking down from the root: at each **duplication** node
we recurse into children; at **speciation** nodes (or leaves) we collect the
entire subtree as one subfamily.

In [ ]:
def extract_subfamilies(node):
    """Recursively split tree at duplication nodes → list of leaf-name lists."""
    if node.is_leaf:
        return [[node.name]]
    if node.props.get("evoltype") == "D":
        clades = []
        for child in node.children:
            clades.extend(extract_subfamilies(child))
        return clades
    else:
        # Speciation node: this whole subtree is one ortholog group
        return [[l.name for l in node.leaves()]]

raw_clades = extract_subfamilies(t)

# Merge very small clades (< 3 leaves) — likely annotation artifacts
subfamilies = [c for c in raw_clades if len(c) >= 3]
singletons = [c for c in raw_clades if len(c) < 3]

print(f"Extracted {len(subfamilies)} clades with ≥3 leaves")
print(f"  ({len(singletons)} small fragments with <3 leaves, discarded)")

### Label subfamilies using gene names (majority vote)

In [ ]:
def label_clade(leaf_names):
    """Assign a subfamily label by majority vote from known gene names."""
    genes = []
    for name in leaf_names:
        g = seqid2gene.get(name, "")
        m = re.search(r'SLC26A(\d+)', g)
        if m:
            genes.append(f"SLC26A{m.group(1)}")
    if genes:
        return Counter(genes).most_common(1)[0][0]
    return "unknown"

subfamily_dict = {}  # label → list of leaf names
for clade in subfamilies:
    label = label_clade(clade)
    # Handle duplicates (merge clades with same label)
    if label in subfamily_dict:
        subfamily_dict[label].extend(clade)
    else:
        subfamily_dict[label] = list(clade)

print(f"{'Subfamily':<12s} {'Sequences':>10s}")
print("-" * 25)
for sf in sorted(subfamily_dict, key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 99):
    print(f"  {sf:<12s} {len(subfamily_dict[sf]):>10d}")

### ✏️ Exercise 2

1. How many subfamilies have exactly one gene per species (1:1 orthologs)?
2. Which subfamilies have **extra copies** in some species (lineage-specific duplications)?
3. Which species have **missing** subfamilies?

In [ ]:
# YOUR CODE HERE

---
## 5. Phylogenetic profile on the NCBI species tree

A **phylogenetic profile** shows gene counts per species per subfamily,
displayed alongside the species phylogeny. This reveals gene losses,
expansions, and conserved 1:1 orthology.

In [ ]:
# Build profile DataFrame
all_taxids = sorted(species_names.keys())

profile = pd.DataFrame(0, index=all_taxids,
                       columns=sorted(subfamily_dict.keys(),
                                      key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 99))

for sf, leaves in subfamily_dict.items():
    for name in leaves:
        taxid = name.split(".")[0]
        if taxid in profile.index and sf in profile.columns:
            profile.loc[taxid, sf] += 1

# Add species names
profile.index = [f"{tid} ({species_names.get(tid, '?')})" for tid in all_taxids]
profile

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(profile, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5,
            cbar_kws={"label": "Gene count"}, ax=ax)
ax.set_title("SLC26 phylogenetic profile")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Visualize on the NCBI species tree with ETE4

In [ ]:
from ete4 import PhyloTree as PT4

# Build NCBI species tree
from ete3 import NCBITaxa
ncbi = NCBITaxa()
sp_tree = PT4(ncbi.get_topology([int(t) for t in all_taxids]).write())
sp_tree.set_outgroup(sp_tree.get_midpoint_outgroup())

# Annotate species tree with profile data
sp_tree.annotate_ncbi_taxa()
for leaf in sp_tree.leaves():
    taxid = str(leaf.props.get("taxid", leaf.name))
    for sf in profile.columns:
        # Find the matching row
        matching = [idx for idx in all_taxids if idx == taxid]
        if matching:
            count = int(profile.loc[f"{taxid} ({species_names.get(taxid, '?')})", sf])
            leaf.add_prop(f"n_{sf}", count)

# Profile layout: show colored gene counts
SF_COLS = list(profile.columns)
COLORS = {0: "#EEEEEE", 1: "#85B7EB", 2: "#D85A30", 3: "#8B0000"}

def profile_layout(node):
    if not node.is_leaf:
        return
    faces = []
    for i, sf in enumerate(SF_COLS):
        n = node.props.get(f"n_{sf}", 0)
        color = COLORS.get(n, "#8B0000")
        faces.append(TextFace(str(n), style={"fill": color, "font-size": "12px"},
                              column=i, position="aligned", padding_x=8))
    return faces

# Header layout
def header_layout(tree_style):
    tree_style.aligned_panel_header = [
        TextFace(sf, style={"fill": "black", "font-size": "10px", "font-weight": "bold"},
                 column=i, position="aligned", padding_x=4)
        for i, sf in enumerate(SF_COLS)
    ]

layouts_profile = [
    Layout(name="Profile", draw_node=profile_layout, active=True),
    Layout(name="Species", draw_node=lambda n: [TextFace(n.props.get("sci_name", n.name),
           style={"fill": "black"}, column=0, position="right")] if n.is_leaf else None),
]

print("Launching phylogenetic profile in ETE4...")
sp_tree.explore(layouts=layouts_profile, keep_server=True, port=5007,
                show_leaf_name=False, name="SLC26 profile")

### ✏️ Exercise 3

From the profile:
1. Which subfamilies are **universally present** (1 copy in all 30 species)?
2. Which species has the **most missing** subfamilies? Why?
3. SLC26A7 has 14 copies in cow (*Bos taurus*) — what might explain this?

In [ ]:
# YOUR CODE HERE

---
## 6. Per-subfamily alignment statistics and trees

We pick 5 representative subfamilies, align each independently,
build trees, and compare statistics.

In [ ]:
SELECTED = ["SLC26A1", "SLC26A3", "SLC26A5", "SLC26A6", "SLC26A8"]
all_seqs = read_fasta(FASTA)

def build_subfamily_tree(label, leaf_names):
    """MAFFT → trimAl → FastTree for a subfamily. Returns (tree_path, trim_path)."""
    fa = os.path.join(SUB_DIR, f"{label}.fa")
    aln_f = os.path.join(SUB_DIR, f"{label}.mafft.fa")
    trim_f = os.path.join(SUB_DIR, f"{label}.mafft.gt01.fa")
    tree_f = os.path.join(SUB_DIR, f"{label}.mafft.gt01.fasttree.nwk")

    # Write subfamily FASTA
    sub_seqs = {n: all_seqs[n] for n in leaf_names if n in all_seqs}
    write_fasta(sub_seqs, fa)

    # Align
    if not os.path.exists(aln_f):
        with open(aln_f, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", fa],
                           stdout=f, stderr=subprocess.PIPE, check=True)
    # Trim
    if not os.path.exists(trim_f):
        subprocess.run(["trimal", "-in", aln_f, "-out", trim_f, "-gt", "0.1", "-fasta"],
                       check=True, capture_output=True)
    # Tree
    if not os.path.exists(tree_f):
        with open(tree_f, "w") as f:
            subprocess.run(["FastTree", "-lg", "-quiet", trim_f],
                           stdout=f, check=True)
    return tree_f, trim_f

# Build all
results = {}
for sf in SELECTED:
    if sf in subfamily_dict:
        tree_f, trim_f = build_subfamily_tree(sf, subfamily_dict[sf])
        results[sf] = {"tree": tree_f, "trim": trim_f}
        a = read_fasta(trim_f)
        print(f"  {sf}: {len(a)} seqs × {len(next(iter(a.values())))} cols")

### 6.1 Compare alignment statistics

In [ ]:
stats = []
for sf, paths in results.items():
    a = read_fasta(paths["trim"])
    n_seqs = len(a)
    aln_len = len(next(iter(a.values())))

    # Mean pairwise identity (estimate from 30 random pairs)
    import random; random.seed(42)
    ids_list = list(a.keys())
    identities = []
    for _ in range(min(30, n_seqs * (n_seqs - 1) // 2)):
        i, j = random.sample(ids_list, 2)
        s1, s2 = a[i], a[j]
        matches = sum(1 for x, y in zip(s1, s2) if x == y and x != "-")
        aligned = sum(1 for x, y in zip(s1, s2) if x != "-" and y != "-")
        if aligned > 0:
            identities.append(matches / aligned)

    # Mean branch length from tree
    pt = PhyloTree(open(paths["tree"]).read())
    branch_lengths = [n.dist for n in pt.traverse() if n.dist > 0]

    stats.append({
        "Subfamily": sf,
        "Sequences": n_seqs,
        "Aln length": aln_len,
        "Mean identity": f"{np.mean(identities):.1%}" if identities else "?",
        "Mean branch length": f"{np.mean(branch_lengths):.4f}",
        "Total tree length": f"{sum(branch_lengths):.3f}",
    })

df_stats = pd.DataFrame(stats).set_index("Subfamily")
df_stats

### ✏️ Exercise 4

1. Which subfamily has the **highest** pairwise identity? The lowest?
2. Is there a relationship between alignment length and tree length?
3. What does a longer total tree length suggest about evolutionary rate?

In [ ]:
# YOUR CODE / NOTES HERE

---
## 7. Method comparison on prestin (SLC26A5)

Distance-based methods (NJ) are more susceptible to **long-branch attraction**
and **convergent substitutions**. If echolocation drives convergent amino acid
changes in prestin, NJ may group echolocators together.

In [ ]:
prestin_fasta = os.path.join(SUB_DIR, "SLC26A5.fa")
prestin_trim = results["SLC26A5"]["trim"]

def build_tree_method(trim_path, method, prefix):
    """Build tree from trimmed alignment using a given method."""
    tree_f = f"{prefix}.{method}.nwk"
    if os.path.exists(tree_f):
        return tree_f

    if method == "fasttree":
        with open(tree_f, "w") as f:
            subprocess.run(["FastTree", "-lg", "-quiet", trim_path],
                           stdout=f, check=True)

    elif method == "iqtree":
        pfx = tree_f.replace(".nwk", "")
        subprocess.run(["iqtree2", "-s", trim_path, "--prefix", pfx,
                        "-mset", "LG", "-B", "1000", "-T", "AUTO", "--quiet"],
                       check=True, capture_output=True)
        if os.path.exists(pfx + ".treefile"):
            os.rename(pfx + ".treefile", tree_f)

    elif method == "nj":
        from Bio import AlignIO
        from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
        from Bio import Phylo
        a = AlignIO.read(trim_path, "fasta")
        dm = DistanceCalculator("blosum62").get_distance(a)
        nj_tree = DistanceTreeConstructor().nj(dm)
        buf = io.StringIO()
        Phylo.write(nj_tree, buf, "newick")
        with open(tree_f, "w") as f:
            f.write(buf.getvalue())

    return tree_f

# Build prestin trees
prefix = os.path.join(SUB_DIR, "SLC26A5")
prestin_trees = {}
for method in ["fasttree", "iqtree", "nj"]:
    try:
        tree_f = build_tree_method(prestin_trim, method, prefix)
        prestin_trees[method] = tree_f
        print(f"  {method}: {tree_f}")
    except Exception as e:
        print(f"  ⚠ {method}: {e}")

### Fill in echolocating taxids

You need to complete this from Exercise 1.

In [ ]:
ECHOLOCATING_TAXIDS = {
    # Toothed whales — fill in
    "9739",     # Tursiops truncatus
    # Add more cetaceans and echolocating bats from your species classification
}

if len(ECHOLOCATING_TAXIDS) < 5:
    print("⚠ Complete ECHOLOCATING_TAXIDS with your bat/cetacean taxids!")

### Explore each tree with echolocators highlighted

In [ ]:
for method, tree_f in prestin_trees.items():
    print(f"\n{'='*50}")
    print(f"  Prestin — {method}")
    print(f"{'='*50}")

    pt = PhyloTree(open(tree_f).read(),
                   sp_naming_function=lambda n: n.split('.')[0])
    pt.set_outgroup(pt.get_midpoint_outgroup())
    pt.annotate_ncbi_taxa()

    def make_echo_layout(echo_taxids):
        def echo_layout(node):
            if not node.is_leaf:
                return
            taxid = node.name.split(".")[0]
            is_echo = taxid in echo_taxids
            sci = node.props.get("sci_name", taxid)
            color = "#D85A30" if is_echo else "#888888"
            weight = "bold" if is_echo else "normal"
            return [TextFace(sci, style={"fill": color, "font-weight": weight},
                             column=0, position="right")]
        return echo_layout

    port = 5010 + list(prestin_trees.keys()).index(method)
    layouts = [Layout(name="Echolocators",
                      draw_node=make_echo_layout(ECHOLOCATING_TAXIDS))]

    pt.explore(layouts=layouts, keep_server=True, port=port,
               show_leaf_name=False, name=f"Prestin — {method}",
               include_props=("name", "sci_name", "dist", "support"))

### ✏️ Exercise 5

Compare the three prestin trees:
1. In which trees do echolocating bats cluster with the **dolphin**?
2. Which method is **most susceptible** to convergent grouping? Why?
3. Where do **fruit bats** (Pteropodidae, non-echolocating) fall in each tree?

### ✏️ Exercise 6

Build the same set of trees for a **control subfamily** (e.g., SLC26A3).
Do echolocators cluster together in those trees?

In [ ]:
# YOUR CODE HERE

---
## Summary

- **ETE4 smartview**: interactive tree exploration in the browser
- **Subfamily extraction**: walk down from root, split at duplication nodes → ~11 ortholog groups
- **Phylogenetic profile**: gene count per species per subfamily reveals losses, expansions, and conserved 1:1 orthology
- **Alignment statistics**: subfamilies differ in conservation, evolutionary rate, and alignment quality
- **Method comparison**: NJ is most susceptible to convergent signal; ML methods (FastTree, IQ-TREE) are more robust

**Tomorrow (Day 2):** We systematically detect convergent residues in prestin, test significance with permutation tests, and compare to control subfamilies.